# Config AWS connection

In [0]:
AWS_ACCESS_KEY = "your-access-key"
AWS_SECRET_KEY = "your-secret-key"
BUCKET_NAME = "your-bucket-name"

spark._jsc.hadoopConfiguration("fs.s3a.access.key", AWS_ACCESS_KEY)
spark._jsc.hadoopConfiguration("fs.s3a.secret.key", AWS_SECRET_KEY)
spark._jsc.hadoopConfiguration("fs.s3a.endpoint", "s3.amazonaws.com")
spark._jsc.hadoopConfiguration("fs.s3a.path.style.access", "true")
spark._jsc.hadoopConfiguration("fs.s3a.connection.ssl.enabled", "false")

# Define Ingestion Configuration

In [0]:
BUCKET  = "s3a://databricks-lakehouse-sources-248"  

INGESTION_CONFIG = [
    {
        "source": "crm",
        "path": f"{BUCKET}/source_crm/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": f"{BUCKET}/source_crm/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": f"{BUCKET}/source_crm/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": f"{BUCKET}/source_erp/CUST_AZ12.csv",
        "table": "erp_cust_az12"
    },
    {
        "source": "erp",
        "path": f"{BUCKET}/source_erp/LOC_A101.csv",
        "table": "erp_loc_a101"
    },
    {
        "source": "erp",
        "path": f"{BUCKET}/source_erp/PX_CAT_G1V2.csv",
        "table": "erp_px_cat_g1v2"
    }
]

# Ingest Files into S3

In [0]:
for item in INGESTION_CONFIG:
    print(f"Ingesting {item['source']} → workspace.bronze.{item['table']}")

    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(item["path"])
    )

    (
        df.write
          .mode("overwrite")
          .format("delta")
          .saveAsTable(f"workspace.bronze.{item['table']}")
    )